In [ ]:
# 1. create a mcp client
# 2. have it spawn a server
# 3. collect the tools that the server can use

# openai agents sdk
# playwright from microsoft
# file tools from anthropic


In [ ]:
# The imports

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os

In [ ]:
load_dotenv(override=True)

In [ ]:
# params is the way of defining mcp server
# uvx - runs a python process that installs packages, this cmd will be run on cmd line that will spawn the mcp server
# args - is the repo from where it should install

In [ ]:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    fetch_tools = await server.list_tools()

fetch_tools 

# this creates a mcp client, that spawns this server fetch, and list out the tools, that i can then give to my agent
# this is python based mcp server, uvx 

In [ ]:
# javascript based mcp server

# fetch also uses playwright only, but here we get lot many functions

# WSL: use Linux npx, not Windows Node from /mnt/c/... (otherwise MCP exits with "Connection closed")
node_bin = os.path.expanduser("~/.nvm/versions/node/v22.22.3/bin")
if os.path.isdir(node_bin):
    os.environ["PATH"] = node_bin + os.pathsep + os.environ["PATH"]

playwright_params = {"command": "npx", "args": ["-y", "@playwright/mcp@latest"]}

async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as server:
    playwright_tools = await server.list_tools()

playwright_tools

In [ ]:
# file server, but tools will only be limited to sandbox folder

sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

async with MCPServerStdio(params=files_params,client_session_timeout_seconds=60) as server:
    file_tools = await server.list_tools()

file_tools

In [ ]:
# agent with mcp servers passed into it

instructions = """
You browse the internet to accomplish your instructions.
You are highly capable at browsing the internet independently to accomplish your task, 
including accepting all cookies and clicking 'not now' as
appropriate to get to the content you need. If one website isn't fruitful, try another. 
Be persistent until you have solved your assignment,
trying different options and sites as needed.
When you need to write files, you do that inside the sandbox folder only.
"""


async with MCPServerStdio(params=files_params, client_session_timeout_seconds=60) as mcp_server_files:
    async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as mcp_server_browser:
        agent = Agent(
            name="investigator", 
            instructions=instructions, 
            model="gpt-4.1-mini",
            mcp_servers=[mcp_server_files, mcp_server_browser]
            )
        with trace("investigate"):
            result = await Runner.run(agent, "Find a great recipe for Banoffee Pie, then summarize it in markdown to banoffee.md")
            print(result.final_output)
